<a href="https://colab.research.google.com/github/ReservaAraras/ECO-SIM/blob/main/SVGtoGIF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Converter SVG Animado para GIF em Loop no Google Drive
# @markdown Defina o tempo de gravação da animação (em segundos) e a taxa de quadros (FPS):
TEMPO_ANIMACAO_SEGUNDOS = 3 # @param {type:"integer"}
FPS = 15 # @param {type:"integer"}

print("Instalando motor de navegador para capturar animações (isso leva cerca de 1 minuto)...")
!pip install playwright google-api-python-client > /dev/null 2>&1
!playwright install chromium > /dev/null 2>&1
!playwright install-deps > /dev/null 2>&1

import os
import io
import time
import asyncio
from PIL import Image
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from playwright.async_api import async_playwright

# 1. Autenticação no Google Drive
print("Solicitando autenticação do Google Drive...")
auth.authenticate_user()
drive_service = build('drive', 'v3')

folder_id = '1t6uOtPiF0Fh6Utbgt4xq-LWSxC7BlsKC'

# Função assíncrona para gravar o SVG rodando no navegador
async def record_animated_svg(svg_path, gif_path, duration, fps):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Abre o SVG no navegador invisível
        await page.goto(f"file://{os.path.abspath(svg_path)}")

        # Aguarda o elemento SVG carregar
        try:
            await page.wait_for_selector('svg', timeout=5000)
            element = await page.query_selector('svg')
        except:
            element = await page.query_selector('body')

        # Pequena pausa para garantir que a animação começou
        await asyncio.sleep(0.5)

        frames = []
        durations =[]
        target_delay = 1.0 / fps
        num_frames = int(duration * fps)

        for _ in range(num_frames):
            start_time = time.time()

            # Tira print do frame atual com fundo transparente
            screenshot_bytes = await element.screenshot(type='png', omit_background=True)
            img = Image.open(io.BytesIO(screenshot_bytes))

            # Adiciona um fundo branco para evitar bugs visuais no GIF
            background = Image.new('RGB', img.size, (255, 255, 255))
            rgba_img = img.convert('RGBA')
            background.paste(rgba_img, mask=rgba_img.split()[3])

            frames.append(background)

            # Calcula o tempo de espera para manter o FPS constante
            elapsed = time.time() - start_time
            sleep_time = max(0.01, target_delay - elapsed)
            await asyncio.sleep(sleep_time)

            # Registra a duração real deste frame para o GIF
            actual_duration = time.time() - start_time
            durations.append(int(actual_duration * 1000))

        await browser.close()

        # Salva os frames como um GIF animado em loop contínuo (loop=0)
        frames[0].save(
            gif_path,
            format='GIF',
            append_images=frames[1:],
            save_all=True,
            duration=durations,
            loop=0
        )

# 2. Buscar todos os arquivos na pasta
print("Buscando arquivos na pasta...")
query = f"'{folder_id}' in parents and trashed=false"
items =[]
page_token = None

while True:
    results = drive_service.files().list(
        q=query, fields="nextPageToken, files(id, name, mimeType)",
        pageToken=page_token, pageSize=1000
    ).execute()
    items.extend(results.get('files',[]))
    page_token = results.get('nextPageToken')
    if not page_token:
        break

svg_files =[f for f in items if f['name'].lower().endswith('.svg') or f.get('mimeType') == 'image/svg+xml']

if not svg_files:
    print('Nenhum arquivo SVG encontrado na pasta.')
else:
    print(f'{len(svg_files)} arquivo(s) SVG encontrado(s). Iniciando gravação das animações...\n')

    for item in svg_files:
        file_id = item['id']
        file_name = item['name']
        print(f'Processando animação: {file_name}')

        local_svg_path = f'/content/{file_name}'
        gif_filename = file_name.rsplit('.', 1)[0] + '.gif'
        gif_path = f'/content/{gif_filename}'

        try:
            # 3. Download do arquivo SVG
            request = drive_service.files().get_media(fileId=file_id)
            svg_content = io.BytesIO()
            downloader = MediaIoBaseDownload(svg_content, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()

            # Salva o SVG temporariamente no Colab para o navegador poder ler
            with open(local_svg_path, 'wb') as f:
                f.write(svg_content.getvalue())

            # 4. Grava a animação e converte para GIF (usando await direto na célula)
            await record_animated_svg(local_svg_path, gif_path, TEMPO_ANIMACAO_SEGUNDOS, FPS)

            # 5. Upload do GIF de volta para o Drive
            file_metadata = {'name': gif_filename, 'parents': [folder_id]}
            media = MediaFileUpload(gif_path, mimetype='image/gif')
            drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

            print(f'  -> GIF animado salvo com sucesso: {gif_filename}')

            # Limpeza dos arquivos temporários
            os.remove(local_svg_path)
            os.remove(gif_path)

        except Exception as e:
            print(f'  -> Erro ao processar {file_name}: {e}')

print('\nProcesso concluído!')

Instalando motor de navegador para capturar animações (isso leva cerca de 1 minuto)...
Solicitando autenticação do Google Drive...


Buscando arquivos na pasta...
92 arquivo(s) SVG encontrado(s). Iniciando gravação das animações...

Processando animação: notebook_80.svg
  -> GIF animado salvo com sucesso: notebook_80.gif
Processando animação: notebook_75.svg
  -> GIF animado salvo com sucesso: notebook_75.gif
Processando animação: notebook_78.svg
  -> GIF animado salvo com sucesso: notebook_78.gif
Processando animação: notebook_79.svg
  -> GIF animado salvo com sucesso: notebook_79.gif
Processando animação: notebook_74.svg
  -> GIF animado salvo com sucesso: notebook_74.gif
Processando animação: notebook_77.svg
  -> GIF animado salvo com sucesso: notebook_77.gif
Processando animação: notebook_76.svg
  -> GIF animado salvo com sucesso: notebook_76.gif
Processando animação: notebook_73.svg
  -> GIF animado salvo com sucesso: notebook_73.gif
Processando animação: notebook_66.svg
  -> GIF animado salvo com sucesso: notebook_66.gif
Processando animação: notebook_72.svg
  -> GIF animado salvo com sucesso: notebook_72.gif
